# Module 6.2 — Debugging & Profiling

### Python Mastery · Track 6: Testing, Tooling & DevOps

When code misbehaves or runs slowly, you need ways to look inside it. This module covers the interactive debugger and `breakpoint()`, structured logging as a durable alternative to scattered prints, and profiling tools that measure where time is actually spent.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- The debugger (`pdb`) is interactive and pauses for input, which cannot run unattended in a notebook. Its commands are therefore presented as a reference, with a runnable post-mortem example and a `.py` script you can run yourself. Logging and profiling run directly in cells.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `breakpoint()` and know the core `pdb` commands.
2. Inspect an error after the fact with post-mortem debugging.
3. Replace ad-hoc prints with the `logging` module, using levels and formatting.
4. Profile CPU time with `cProfile` to find bottlenecks.
5. Benchmark small snippets accurately with `timeit`.

**Prerequisites:** Tracks 1 to 5.

---

## Part 1 · The Debugger: `breakpoint()` and `pdb`

The built-in debugger lets you pause a running program and inspect it line by line. You insert a pause with `breakpoint()` (which starts `pdb`), then issue commands at the `(Pdb)` prompt.

Because the debugger waits for typed input, it cannot run unattended in a notebook cell, so do not call `breakpoint()` here; it would hang. Instead, learn the essential commands, which you will use in a terminal or in your editor's debugger:

| Command | Meaning |
|---|---|
| `n` (next) | run the current line, step **over** function calls |
| `s` (step) | step **into** a function call |
| `c` (continue) | resume until the next breakpoint or the end |
| `l` (list) | show source around the current line |
| `p expr` | print the value of an expression |
| `pp expr` | pretty-print a value |
| `w` (where) | show the call stack |
| `q` (quit) | abort the program |

The next cell writes a small script that uses `breakpoint()`; you can run it yourself in a terminal with `python debug_demo.py` to practise the commands.

In [ ]:
%%writefile debug_demo.py
def average(numbers):
    total = sum(numbers)
    count = len(numbers)
    # breakpoint()   # <- uncomment, then run `python debug_demo.py` in a terminal
    return total / count

if __name__ == "__main__":
    print(average([10, 20, 30]))

In [ ]:
# We can run the script normally here (the breakpoint is commented out).
# To practise pdb, uncomment the breakpoint line and run it in a terminal.
!python debug_demo.py

## Part 2 · Post-Mortem Debugging

When an exception occurs, you can examine the program's state **at the moment it failed**, without rerunning it, using post-mortem debugging. Interactively this is `pdb.post_mortem()`. Non-interactively, the `traceback` module lets you inspect and format the failure, which is what the cell below demonstrates so it runs unattended.

In [ ]:
import traceback

def process(data):
    return data["value"] * 2          # will fail if 'value' is missing

try:
    process({"name": "test"})         # no 'value' key -> KeyError
except Exception:
    # Capture and display the traceback as text, as a logger or report would.
    formatted = traceback.format_exc()
    print("an error occurred; here is the traceback:")
    print(formatted.strip())

## Part 3 · Logging Instead of Printing

Scattering `print` calls to understand a program is tempting but fragile: prints are easy to forget, hard to switch off, and carry no context. The `logging` module is the durable alternative. Messages have **levels** (DEBUG, INFO, WARNING, ERROR, CRITICAL), so you can control how much detail appears, and you can format each record with a timestamp, level, and source. In libraries and applications, logging is the standard.

In [ ]:
import logging

# Configure the root logger once: set the threshold and the message format.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True,                       # reconfigure even if already configured (useful in notebooks)
)

log = logging.getLogger("demo")

log.debug("detailed diagnostic information")
log.info("normal operational message")
log.warning("something unexpected but not fatal")
log.error("a serious problem occurred")
# Each line shows its level, so you can filter by importance.

The **level** acts as a filter. Setting the threshold to `WARNING` suppresses DEBUG and INFO messages without removing them from the code, so you can raise or lower verbosity for development versus production by changing one setting.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("filtered")

log.debug("you will NOT see this")          # below the threshold
log.info("nor this")                         # below the threshold
log.warning("but you WILL see this")         # at the threshold
log.error("and this")                        # above the threshold

In [ ]:
import logging

# Logging is excellent for recording exceptions with their traceback.
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("errors")

def risky(n):
    return 10 / n

try:
    risky(0)
except ZeroDivisionError:
    # exc_info=True attaches the traceback to the log record.
    log.error("calculation failed", exc_info=True)

## Part 4 · Profiling with `cProfile`

Before optimising, **measure**. Guessing where a program spends its time is unreliable; profiling tells you. The standard `cProfile` records how often each function is called and how long it takes, so you can focus effort on the genuine bottleneck. The cell below profiles a function and prints the busiest calls.

In [ ]:
import cProfile
import pstats
import io

def slow_part():
    return sum(i * i for i in range(100_000))

def fast_part():
    return sum(range(100))

def program():
    total = 0
    for _ in range(10):
        total += slow_part()        # called repeatedly; likely the bottleneck
    total += fast_part()
    return total

# Profile the program and capture the statistics.
profiler = cProfile.Profile()
profiler.enable()
program()
profiler.disable()

# Report the functions that took the most cumulative time.
buffer = io.StringIO()
stats = pstats.Stats(profiler, stream=buffer).sort_stats("cumulative")
stats.print_stats(5)                 # top 5 by cumulative time
print(buffer.getvalue())

The report makes the hot spot obvious: `slow_part`, called ten times, dominates the runtime, while `fast_part` is negligible. This is the information you need to optimise wisely, rather than guessing. For line-by-line detail, the third-party `line_profiler` tool drills further into a single function, though `cProfile` is usually the right first step.

## Part 5 · Micro-Benchmarking with `timeit`

To compare two small pieces of code fairly, use `timeit`, which runs a snippet many times and reports the total, smoothing out noise. It is the right tool for questions like "is a list comprehension faster than a loop here?" Beware of drawing big conclusions from tiny differences; measure realistic code.

In [ ]:
import timeit

# Compare two ways of building a list of squares, each run many times.
loop_time = timeit.timeit(
    """
result = []
for i in range(1000):
    result.append(i * i)
""",
    number=1000,
)

comprehension_time = timeit.timeit(
    "result = [i * i for i in range(1000)]",
    number=1000,
)

print(f"loop:          {loop_time:.4f}s for 1000 runs")
print(f"comprehension: {comprehension_time:.4f}s for 1000 runs")
print("the comprehension is typically faster for this kind of work")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Basic logging (Easy)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("ex1")

log.info("starting up")
log.warning("low disk space")
log.info("shutting down")

### Example 2 — Capturing a traceback as text (Easy)

In [ ]:
import traceback

try:
    [1, 2, 3][10]                # IndexError
except Exception:
    print("caught and recorded:")
    print(traceback.format_exc().strip().splitlines()[-1])   # the final summary line

### Example 3 — Filtering by log level (Medium)

In [ ]:
import logging
logging.basicConfig(level=logging.ERROR, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("ex3")

messages = [
    (log.debug, "debug detail"),
    (log.info, "info note"),
    (log.warning, "a warning"),
    (log.error, "an error"),
]
for func, text in messages:
    func(text)                    # only ERROR and above will appear
print("(only the ERROR message was shown above)")

### Example 4 — Profiling to find the bottleneck (Medium)

In [ ]:
import cProfile, pstats, io

def cheap():
    return sum(range(1000))

def expensive():
    return sum(i ** 2 for i in range(200_000))

def run():
    cheap()
    expensive()
    cheap()

profiler = cProfile.Profile()
profiler.enable()
run()
profiler.disable()

buf = io.StringIO()
pstats.Stats(profiler, stream=buf).sort_stats("tottime").print_stats(4)
# The function with the largest 'tottime' is the bottleneck.
print(buf.getvalue())

### Example 5 — Comparing algorithms with timeit (Difficult)

In [ ]:
import timeit

# Membership test: list versus set, for a value that is not present.
setup_list = "data = list(range(1000))"
setup_set = "data = set(range(1000))"
check = "999 in data"

list_time = timeit.timeit(check, setup=setup_list, number=100_000)
set_time = timeit.timeit(check, setup=setup_set, number=100_000)

print(f"list membership: {list_time:.4f}s")
print(f"set membership:  {set_time:.4f}s")
print("set membership is far faster, as it does not scan every element")

### Example 6 — A reusable timing context manager (Difficult)

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    """Measure how long the enclosed block takes."""
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label}: {elapsed:.4f}s")

with timed("building a big list"):
    data = [i * i for i in range(1_000_000)]

with timed("summing it"):
    total = sum(data)

print("sum computed:", total > 0)

---

## Exercises

**Exercise 1 (Easy).** Configure logging at the INFO level and emit one INFO message and one WARNING message. Confirm both appear.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Set the logging level to WARNING, then emit a DEBUG, an INFO, and a WARNING message. Only the WARNING should appear.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Catch a `ValueError` raised by `int("oops")` and log it at ERROR level with the traceback attached (use `exc_info=True`).

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `timeit` to compare summing a list with a `for` loop accumulator versus the built-in `sum`, over `range(1000)`, each run many times. Print both timings.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Use `cProfile` to profile a function that calls a cheap helper and an expensive helper, and print the top three functions sorted by total time (`tottime`).

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("sol1")
log.info("an info message")
log.warning("a warning message")

**Solution 2**

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("sol2")
log.debug("hidden")
log.info("hidden")
log.warning("shown")

**Solution 3**

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("sol3")
try:
    int("oops")
except ValueError:
    log.error("conversion failed", exc_info=True)

**Solution 4**

In [ ]:
import timeit
loop = timeit.timeit("""
total = 0
for i in range(1000):
    total += i
""", number=10_000)
builtin = timeit.timeit("total = sum(range(1000))", number=10_000)
print(f"loop:    {loop:.4f}s")
print(f"builtin: {builtin:.4f}s")

**Solution 5**

In [ ]:
import cProfile, pstats, io

def cheap():
    return sum(range(100))

def expensive():
    return sum(i * i for i in range(300_000))

def run():
    cheap()
    expensive()

profiler = cProfile.Profile()
profiler.enable()
run()
profiler.disable()

buf = io.StringIO()
pstats.Stats(profiler, stream=buf).sort_stats("tottime").print_stats(3)
print(buf.getvalue())

---

## Summary and Key Points

- `breakpoint()` starts the `pdb` debugger; the core commands are `n` (over), `s` (into), `c` (continue), `l` (list), `p`/`pp` (print), `w` (stack), and `q` (quit). The debugger is interactive, so practise it in a terminal rather than a notebook.
- Post-mortem debugging inspects state at the point of failure; the `traceback` module formats and records the failure non-interactively.
- The `logging` module replaces ad-hoc prints: messages carry levels (DEBUG to CRITICAL) you can filter, plus formatting and timestamps, and `exc_info=True` attaches a traceback.
- Profile before optimising: `cProfile` reports calls and time per function, revealing the true bottleneck; sort by cumulative or total time.
- `timeit` benchmarks small snippets by running them many times; compare realistic code and be wary of tiny differences.

### Next module: 6.3 — Type Hints & Static Checking

The next module covers annotating code with types and checking those annotations with `mypy`, including generics, `Optional`, `Protocol`, and `TypedDict`.